# Build Top and Rising Event Word Dictionaries

This notebook reads all event-word CSV files in the working directory, builds a global **top query dictionary** by counting query frequency across event files, and builds a **rising query dictionary** by aggregating frequency and search-interest statistics. Outputs are saved to `event_words/`.

In [ ]:
from pathlib import Path
import re
import pandas as pd

# Change this if your CSV files live somewhere else.
DATA_DIR = Path(".")
OUTPUT_DIR = Path("event_words")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TOP_PREFIX = "searched_with_top-searches_US"
RISING_PREFIX = "searched_with_rising-searches_US"

TOP_PATTERN = f"{TOP_PREFIX}*.csv"
RISING_PATTERN = f"{RISING_PREFIX}*.csv"

In [ ]:
def clean_query(s: pd.Series) -> pd.Series:
    """Normalize query strings for matching across files."""
    return (
        s.astype("string")
         .str.strip()
         .str.lower()
         .str.replace(r"\s+", " ", regex=True)
    )


def parse_increase_percent(s: pd.Series) -> pd.Series:
    """Convert strings like '4,150%' or '-8%' to numeric percentage points."""
    return pd.to_numeric(
        s.astype("string")
         .str.replace("%", "", regex=False)
         .str.replace(",", "", regex=False)
         .str.strip(),
        errors="coerce"
    )


def event_date_from_filename(path: Path) -> str:
    """Extract event date from filenames like searched_with_top-searches_US_20240801-2000_...csv."""
    m = re.search(r"US_(\d{8})-", path.name)
    if not m:
        return "unknown"
    return pd.to_datetime(m.group(1), format="%Y%m%d").date().isoformat()


def read_event_files(pattern: str, source_type: str) -> pd.DataFrame:
    """Read and lightly standardize all matching event-word CSV files."""
    files = sorted(DATA_DIR.glob(pattern))
    if not files:
        raise FileNotFoundError(f"No files matched {pattern!r} in {DATA_DIR.resolve()}")

    frames = []
    usecols = ["query", "search interest", "increase percent"]

    for f in files:
        df = pd.read_csv(f, usecols=lambda c: c in usecols)
        df.columns = [c.strip().lower().replace(" ", "_") for c in df.columns]

        df["query_raw"] = df["query"].astype("string").str.strip()
        df["query_clean"] = clean_query(df["query"])
        df["search_interest"] = pd.to_numeric(df["search_interest"], errors="coerce")
        df["increase_percent_num"] = parse_increase_percent(df["increase_percent"])
        df["event_date"] = event_date_from_filename(f)
        df["source_file"] = f.name
        df["source_type"] = source_type

        frames.append(
            df[[
                "source_type", "event_date", "source_file",
                "query_raw", "query_clean", "search_interest",
                "increase_percent", "increase_percent_num"
            ]]
        )

    out = pd.concat(frames, ignore_index=True)
    out = out.dropna(subset=["query_clean"])
    out = out[out["query_clean"].ne("")]
    return out

In [ ]:
top_raw = read_event_files(TOP_PATTERN, "top")
rising_raw = read_event_files(RISING_PATTERN, "rising")

print(f"Top files: {top_raw['source_file'].nunique()}")
print(f"Top rows: {len(top_raw):,}")
print(f"Rising files: {rising_raw['source_file'].nunique()}")
print(f"Rising rows: {len(rising_raw):,}")

In [ ]:
# Top dictionary: count how often each query appears across top-event files.
# Use nunique(source_file), not raw row count, so duplicate rows within one file do not inflate frequency.

top_dict = (
    top_raw
    .groupby("query_clean", as_index=False)
    .agg(
        query=("query_raw", "first"),
        frequency=("source_file", "nunique"),
        n_rows=("query_raw", "size"),
        mean_search_interest=("search_interest", "mean"),
        max_search_interest=("search_interest", "max"),
        min_search_interest=("search_interest", "min"),
        event_dates=("event_date", lambda x: ";".join(sorted(set(x)))),
        source_files=("source_file", lambda x: ";".join(sorted(set(x))))
    )
    .sort_values(["frequency", "mean_search_interest", "query"], ascending=[False, False, True])
    .reset_index(drop=True)
)

top_dict.to_csv(OUTPUT_DIR / "top_dictionary_frequency.csv", index=False)
top_raw.to_csv(OUTPUT_DIR / "top_dictionary_long.csv", index=False)

top_dict.head(20)

In [ ]:
# Rising dictionary: aggregate event-specific rising queries.
# This preserves both frequency and search-interest / increase-percent summaries.

rising_dict = (
    rising_raw
    .groupby("query_clean", as_index=False)
    .agg(
        query=("query_raw", "first"),
        frequency=("source_file", "nunique"),
        n_rows=("query_raw", "size"),
        total_search_interest=("search_interest", "sum"),
        mean_search_interest=("search_interest", "mean"),
        max_search_interest=("search_interest", "max"),
        mean_increase_percent=("increase_percent_num", "mean"),
        max_increase_percent=("increase_percent_num", "max"),
        event_dates=("event_date", lambda x: ";".join(sorted(set(x)))),
        source_files=("source_file", lambda x: ";".join(sorted(set(x))))
    )
    .sort_values(
        ["frequency", "total_search_interest", "max_increase_percent", "query"],
        ascending=[False, False, False, True]
    )
    .reset_index(drop=True)
)

rising_dict.to_csv(OUTPUT_DIR / "rising_dictionary_aggregated.csv", index=False)
rising_raw.to_csv(OUTPUT_DIR / "rising_dictionary_long.csv", index=False)

rising_dict.head(20)

In [ ]:
# Optional: thresholded dictionaries for downstream extraction.
# Adjust these cutoffs based on API budget and research design.

TOP_MIN_FREQUENCY = 2
RISING_MIN_FREQUENCY = 2

stable_top_dict = top_dict[top_dict["frequency"] >= TOP_MIN_FREQUENCY].copy()
global_rising_dict = rising_dict[rising_dict["frequency"] >= RISING_MIN_FREQUENCY].copy()

stable_top_dict.to_csv(OUTPUT_DIR / f"top_dictionary_frequency_ge{TOP_MIN_FREQUENCY}.csv", index=False)
global_rising_dict.to_csv(OUTPUT_DIR / f"rising_dictionary_frequency_ge{RISING_MIN_FREQUENCY}.csv", index=False)

print(f"Stable top terms, frequency >= {TOP_MIN_FREQUENCY}: {len(stable_top_dict):,}")
print(f"Global rising terms, frequency >= {RISING_MIN_FREQUENCY}: {len(global_rising_dict):,}")

In [ ]:
# Optional: local rising dictionaries, one file per event date.
# This is useful because rising queries are usually event-specific.

local_dir = OUTPUT_DIR / "local_rising_by_event_date"
local_dir.mkdir(parents=True, exist_ok=True)

for event_date, df in rising_raw.groupby("event_date", sort=True):
    local = (
        df.sort_values(["search_interest", "increase_percent_num", "query_clean"], ascending=[False, False, True])
          .drop_duplicates("query_clean")
          .reset_index(drop=True)
    )
    local.to_csv(local_dir / f"rising_dictionary_{event_date}.csv", index=False)

print(f"Wrote local rising dictionaries to: {local_dir}")
print("Main outputs:")
for p in sorted(OUTPUT_DIR.glob("*.csv")):
    print(" -", p)